# 🎯 [QLT04_M10] 전술 장비 가동률 저하 요인 분석 및 가동률 극대화 도구 최종 완성 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/QLT04/qlt04_m10_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 품질·규격·검사 (Level 2) |
| **미션 ID** | QLT04_M10 |
| **미션 주제** | 전술 장비 가동률 저하 요인 분석 및 가동률 극대화 도구 최종 완성 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
외주정비는 정비 대기, 외주 검사 대기, 재작업, 규격·제원 불일치가 겹치면서 장비 가동률을 떨어뜨리지만 요인별 영향이 한눈에 보이지 않아 어디에 정비 자원을 먼저 투입할지 정하기 어렵다. 본 프로젝트는 미 국방부 외주정비 품질보증 사례의 문제의식(규격 정합성·검사 병목·하자 연결)을 교육용 가상 자료로 재구성해, 정비 대기 요인 분류, 저가동 요인 분석, 재배치 시뮬레이션, 추세 대시보드, 사람 검토 워크플로우까지 프로토타입으로 구현하는 과제이다. 가상 장비는 유형 A·B·C와 가상 부대로만 구분하며 실제 장비·부대 자료는 쓰지 않는다. 최종 목표는 가동률 저하 요인과 외주 품질검사 병목을 근거와 함께 보여 주되, 정비 우선순위와 자원 투입 결정은 정비관·품질검사관·군수참모가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 프로세스 정의서(`qlt04_as_is_to_be_spec.md`), UAT 피드백 리포트(`qlt04_uat_feedback_report.md`), 운영 매뉴얼(`qlt04_final_manual.md`) 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 품질 판정 룰 엔진, 릿지 회귀 XAI 엔진, 시뮬레이터 UI, HITL 결재 FSM, 배치 러너 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 야전 배포를 위한 최종 패키지(`qlt04_package/`)

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 10. 전술 장비 가동률 저하 요인 분석 및 가동률 극대화 도구 최종 완성
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

전술 장비 가동률 저하 요인 분석 및 가동률 극대화 도구를 완성한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **입력 검증부터 룰 판별, 요인 추천, 사람 검토 큐, 감사 로그, 브리핑까지 하나로 묶은 가동률 저하 요인 분석·극대화 도구를 완성하고 배포 점검을 마친다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **앞 미션의 기능을 통합 파이프라인으로 묶어 설치 스크립트 5단계 점검, 서로 다른 시드 4종의 End-to-End 실행, 자체 점검 9항목과 함수 커버리지 수행하고 분석하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `dist/qlt04_package/, README.md, docs/qlt04_final_manual.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 상용 소프트웨어 수준의 완성도 및 조직 확산 가치

#### 미션 PASS 조건

- 정량 기준: 설치부터 실행까지 원클릭 설치 스크립트 완비, 매뉴얼 작성률 100%


- 진행 기준: 최종 배포본 생성 및 검수 완료


- 검증 방법: Clean 환경에서의 설치 및 초기화 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""입력 검증·격리, 16종 룰 판별, 가동률 회귀와 저하 요인 Top-3 추천, 사람 검토 큐와 해시 체인 감사 로그, 브리핑 요약까지 외주정비 품질검사·가동률 분석 도구를 한 파일로 통합 실행하고 설치 스크립트 점검·End-to-End 4종·자체 점검(함수 커버리지)·매뉴얼 작성률을 출력한다."""
import hashlib
import json
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 장비 가동률·정비 대기 자료 생성기 (실제 부대·장비·제원·가동률 정보 없음)
# 장비 유형 A(궤도형)·B(화포형)·C(차륜형)은 추상 구분이며 실제 장비와 무관하다.
# 최근 90일 동안 장비 1대의 정비 대기 요인(부품·정비병·외주 정비·외주 검사·재작업)과 가동률을 만든다.
# ---------------------------------------------------------------------------
UNITS = ["가상A부대", "가상B부대", "가상C부대", "가상D부대", "가상E부대", "가상F부대"]
ETYPES = ["유형A", "유형B", "유형C"]
WINDOW = 90                                       # 관측 기간(일)
CAUSES = ["부품 대기", "정비병 부족", "외주 정비", "검사·재작업"]
CAUSE_COL = {"부품 대기": "parts_wait_days", "정비병 부족": "labor_wait_days", "외주 정비": "outsource_days", "검사·재작업": "inspect_wait_days"}
NUM_FIELDS = ["age_years", "op_hours", "maint_orders", "parts_wait_days", "labor_wait_days", "outsource_days", "inspect_wait_days", "rework_count",
              "spec_mismatch", "defect_reports", "parts_stock_pct", "mechanic_fill_pct", "avail_pct"]
FEAT = ["age_years", "maint_orders", "parts_stock_pct", "mechanic_fill_pct", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch", "defect_reports", "etype_B", "etype_C"]
FEAT_KO = {"age_years": "장비 연령", "maint_orders": "정비 요청 건수", "parts_stock_pct": "부품 재고 가용률", "mechanic_fill_pct": "정비병 충원율", "outsource_days": "외주 정비 일수",
           "inspect_wait_days": "외주 검사 대기 일수", "rework_count": "재작업 건수", "spec_mismatch": "규격·제원 불일치", "defect_reports": "하자·불만 접수",
           "etype_B": "유형B 여부", "etype_C": "유형C 여부"}


def unit_profile(seed=3):
    """부대별 고정 특성: 부품 재고 수준, 정비병 충원율, 외주 정비소 거리(일수 가중), 외주 검사 혼잡도."""
    r = np.random.default_rng(seed)
    return pd.DataFrame({"unit": UNITS, "stock": r.uniform(45, 95, len(UNITS)), "fill": r.uniform(55, 98, len(UNITS)),
                         "dist": r.uniform(0.8, 1.6, len(UNITS)), "queue": r.uniform(0.7, 2.2, len(UNITS))})


def make_fleet(n=300, seed=7, shock=0.0):
    """장비 n대의 90일 관측 자료(필드 16개)와 정답 요인을 만든다. shock은 외주 검사 병목 악화 정도(AAR용 환경 변화)."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    ui = rng.integers(0, len(UNITS), n)
    et = rng.choice(ETYPES, n, p=[.35, .3, .35])
    age = np.clip(rng.gamma(4, 2.2, n), 0.5, 30).round(1)
    eff = np.array([{"유형A": 1.15, "유형B": 1.0, "유형C": 0.9}[t] for t in et])
    orders = rng.poisson(np.clip(1.2 + 0.12 * age, 0.5, 6) * eff)
    stock = np.clip(pf["stock"].to_numpy()[ui] + rng.normal(0, 8, n), 20, 100).round(0)
    fill = np.clip(pf["fill"].to_numpy()[ui] + rng.normal(0, 6, n), 30, 100).round(0)
    parts_w = (orders * rng.gamma(2, 0.55 * (1 + ((100 - stock) / 28) ** 2) / 2)).round(0)
    labor_w = (orders * rng.gamma(2, 0.7 * (1 + ((100 - fill) / 24) ** 2) / 2)).round(0)
    n_out = rng.binomial(orders, 0.3)
    out_d = (n_out * rng.gamma(3, 4.5 * pf["dist"].to_numpy()[ui] / 3)).round(0)
    insp_w = (n_out * rng.gamma(2, (2.6 + shock) * pf["queue"].to_numpy()[ui] / 2)).round(0)
    spec = rng.poisson(0.2 + 0.04 * age)
    rework = rng.binomial(n_out, np.clip(0.08 + 0.1 * spec, 0, .6))
    defect = rng.poisson(0.3 + 0.7 * rework)
    base_rep = orders * 1.8
    down = base_rep + parts_w + labor_w + out_d + insp_w + rework * 4 + rng.normal(0, 1.2, n)
    comp = np.column_stack([parts_w, labor_w, out_d, insp_w + rework * 4])
    df = pd.DataFrame({"equip_id": [f"E-{i + 1:04d}" for i in range(n)], "unit": np.array(UNITS)[ui], "etype": et, "age_years": age,
                       "op_hours": np.clip(rng.normal(320 - 4 * age, 70, n), 40, 600).round(0), "maint_orders": orders,
                       "parts_wait_days": parts_w, "labor_wait_days": labor_w, "outsource_days": out_d, "inspect_wait_days": insp_w,
                       "rework_count": rework, "spec_mismatch": spec, "defect_reports": defect, "parts_stock_pct": stock, "mechanic_fill_pct": fill,
                       "avail_pct": (100 * (1 - np.clip(down, 0, WINDOW) / WINDOW)).round(1)})
    df["main_cause"] = np.where(comp.max(axis=1) < 3, "없음", np.array(CAUSES)[comp.argmax(axis=1)])
    return df


def feat_frame(df):
    """회귀·분석용 설명 변수 표(유형은 더미 변수)."""
    x = df.copy()
    x["etype_B"], x["etype_C"] = (x["etype"] == "유형B").astype(int), (x["etype"] == "유형C").astype(int)
    return x[FEAT]


def make_trend(months=12, seed=5, shock_unit=None):
    """[교육용가상자료] 부대 x 유형 x 월 평균 가동률 추세. shock_unit 부대는 하반기에 외주 검사 병목으로 가동률이 떨어진다."""
    rng = np.random.default_rng(seed)
    pf = unit_profile()
    rows = []
    for _, u in pf.iterrows():
        base = 62 + 0.22 * u["stock"] + 0.12 * u["fill"] - 3.0 * u["queue"]
        for t in ETYPES:
            off = {"유형A": -3.0, "유형B": 0.0, "유형C": 2.0}[t]
            for m in range(months):
                v = base + off + 2.0 * np.sin(m / 12 * 2 * np.pi) + rng.normal(0, 1.0)
                if shock_unit == u["unit"] and m >= months // 2:
                    v -= 1.5 * (m - months // 2 + 1)
                rows.append((u["unit"], t, m + 1, round(float(np.clip(v, 30, 99)), 1)))
    return pd.DataFrame(rows, columns=["unit", "etype", "month", "avail_pct"])

# ---- 정비·검사 병목 판별 룰 설정(YAML 형식 문자열: 기준이 바뀌면 코드가 아니라 이 설정만 고친다) ----
# clause는 룰과 대응시킨 '가상 품질지침 조항' 예시이며 실제 규정 원문이 아니다.
RULES_YAML = """
- {id: Q01, name: 부품 재고 가용률 50% 미만, field: parts_stock_pct, op: "<", value: 50, weight: 2, clause: 가상품질지침 제3조}
- {id: Q02, name: 정비병 충원율 60% 미만, field: mechanic_fill_pct, op: "<", value: 60, weight: 2, clause: 가상품질지침 제4조}
- {id: Q03, name: 부품 대기 8일 초과, field: parts_wait_days, op: ">", value: 8, weight: 2, clause: 가상품질지침 제5조}
- {id: Q04, name: 정비병 부족 대기 8일 초과, field: labor_wait_days, op: ">", value: 8, weight: 2, clause: 가상품질지침 제6조}
- {id: Q05, name: 외주 정비 15일 초과, field: outsource_days, op: ">", value: 15, weight: 2, clause: 가상품질지침 제7조}
- {id: Q06, name: 외주 검사 대기 6일 초과, field: inspect_wait_days, op: ">", value: 6, weight: 3, clause: 가상품질지침 제8조}
- {id: Q07, name: 재작업 1건 이상, field: rework_count, op: ">=", value: 1, weight: 3, clause: 가상품질지침 제9조}
- {id: Q08, name: 재작업률 25% 초과, field: rework_rate, op: ">", value: 0.25, weight: 2, clause: 가상품질지침 제9조}
- {id: Q09, name: 규격·제원 불일치 3건 이상, field: spec_mismatch, op: ">=", value: 3, weight: 3, clause: 가상품질지침 제10조}
- {id: Q10, name: 하자·불만 3건 이상, field: defect_reports, op: ">=", value: 3, weight: 2, clause: 가상품질지침 제11조}
- {id: Q11, name: 장비 연령 15년 초과, field: age_years, op: ">", value: 15, weight: 1, clause: 가상품질지침 제12조}
- {id: Q12, name: 정비 요청 6건 이상, field: maint_orders, op: ">=", value: 6, weight: 1, clause: 가상품질지침 제13조}
- {id: Q13, name: 대기 일수 합계 30일 초과, field: total_wait, op: ">", value: 30, weight: 3, clause: 가상품질지침 제14조}
- {id: Q14, name: 검사 대기가 외주 정비 일수의 80% 초과, field: insp_ratio, op: ">", value: 0.8, weight: 2, clause: 가상품질지침 제8조}
- {id: Q15, name: 규격 불일치 2건 이상과 하자 접수 동반, field: spec_defect, op: "==", value: true, weight: 3, clause: 가상품질지침 제10조}
- {id: Q16, name: 최근 90일 가동 시간 150시간 미만, field: op_hours, op: "<", value: 150, weight: 1, clause: 가상품질지침 제15조}
"""
GRADE_CUT = [(10, "고위험"), (6, "주의"), (3, "관심")]
OPS = {"<": lambda a, b: a < b, ">": lambda a, b: a > b, "<=": lambda a, b: a <= b, ">=": lambda a, b: a >= b, "==": lambda a, b: a == b}


def parse_rules(text):
    """YAML 한 줄 맵({key: value, ...}) 형식 룰 문자열을 읽는다(외부 yaml 라이브러리 없이)."""
    out = []
    for line in text.strip().splitlines():
        body = line.strip()[3:-1]
        r = {}
        for part in body.split(", "):
            k, v = part.split(": ", 1)
            v = v.strip().strip('"')
            r[k] = True if v == "true" else False if v == "false" else (float(v) if v.replace(".", "", 1).isdigit() else v)
        out.append(r)
    return out


def derive_rec(rec):
    """룰이 쓰는 파생 필드(대기 합계, 재작업률, 검사 대기 비율(외주 5일 이상일 때), 규격 불일치 2건 이상+하자 동반)를 추가한다."""
    r = dict(rec)
    r["total_wait"] = r["parts_wait_days"] + r["labor_wait_days"] + r["outsource_days"] + r["inspect_wait_days"]
    r["rework_rate"] = r["rework_count"] / r["maint_orders"] if r["maint_orders"] else 0.0
    r["insp_ratio"] = r["inspect_wait_days"] / r["outsource_days"] if r["outsource_days"] >= 5 else 0.0
    r["spec_defect"] = bool(r["spec_mismatch"] >= 2 and r["defect_reports"] >= 1)
    return r


def apply_rules(rec, rules, derived=False):
    """룰 적용: (점수, 적중 룰 id 목록). derived=True면 이미 파생 필드가 있는 레코드(경계 시험용)로 본다."""
    r = rec if derived else derive_rec(rec)
    hits = [x["id"] for x in rules if OPS[x["op"]](r[x["field"]], x["value"])]
    w = {x["id"]: x["weight"] for x in rules}
    return sum(w[h] for h in hits), hits


def grade_of(score):
    for cut, g in GRADE_CUT:
        if score >= cut:
            return g
    return "정상"

# ---- 가동률 회귀 모델과 저가동 요인 분류·추천 모델(외부 AI 서비스 없이 scikit-learn 사용) ----
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

CAUSE_COLS = ["age_years", "maint_orders", "parts_stock_pct", "mechanic_fill_pct", "outsource_days", "inspect_wait_days", "rework_count", "spec_mismatch",
              "defect_reports", "parts_wait_days", "labor_wait_days"]
ACTION = {"부품 대기": "부품 사전 청구·재고 보강", "정비병 부족": "정비병 재배치·교육 일정 조정", "외주 정비": "외주 정비 일정 조율·내부 정비 확대", "검사·재작업": "검사 일정 조율·규격 사전 대조"}


def cause_frame(df):
    x = df[CAUSE_COLS].copy()
    x["out_insp"] = x["outsource_days"] + x["inspect_wait_days"] + 4 * x["rework_count"]
    x["wait_pm"] = (x["parts_wait_days"] + x["labor_wait_days"]) / (x["maint_orders"] + 1)
    return x


def fit_avail(df, alpha=1.0):
    """가동률(%) 회귀: 표준화 + 릿지 회귀."""
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(feat_frame(df), df["avail_pct"])


def fit_cause(df, cut=85, C=5):
    """가동률이 cut% 미만인 장비의 주요 저가동 요인(4종)을 분류한다."""
    low = df[(df["avail_pct"] < cut) & (df["main_cause"] != "없음")]
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000)).fit(cause_frame(low), low["main_cause"])


SIM_FEAT = ["age_years", "maint_orders", "spec_mismatch", "stock_gap_x_orders", "fill_gap_x_orders", "parts_stock_pct", "mechanic_fill_pct", "etype_B", "etype_C"]


def sim_frame(df):
    """자원 재배치 시뮬레이션용 설명 변수: 부족분이 클수록 대기가 더 빠르게 늘고 정비 요청 건수와 곱해져 커진다는 구조를 제곱·상호작용 항으로 둔다."""
    x = df.copy()
    x["etype_B"], x["etype_C"] = (x["etype"] == "유형B").astype(int), (x["etype"] == "유형C").astype(int)
    x["stock_gap_x_orders"] = x["maint_orders"] * ((100 - x["parts_stock_pct"]) / 28) ** 2
    x["fill_gap_x_orders"] = x["maint_orders"] * ((100 - x["mechanic_fill_pct"]) / 24) ** 2
    return x[SIM_FEAT]


def fit_sim(df, alpha=0.5):
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(sim_frame(df), df["avail_pct"])

RULES = parse_rules(RULES_YAML)
REQ = {"equip_id": r"^E-\d{4}$", "unit": UNITS, "etype": ETYPES, "avail_pct": (0, 100), "maint_orders": (0, 30), "age_years": (0, 40), "parts_stock_pct": (0, 100), "mechanic_fill_pct": (0, 100)}
CONF = {"queue_min_score": 6}
MANUAL = {
    "1. 목적과 범위": "외주정비 품질검사·정비 대기로 인한 가동률 저하 요인을 찾는 분석 보조 도구다. 우선순위·자원 투입 결정은 정비관·품질검사관·군수참모가 한다.",
    "2. 설치와 실행": "install.sh 한 번으로 환경 점검, 의존 패키지 확인, 자체 시험, 데모 실행을 순서대로 수행한다.",
    "3. 입력 데이터": "장비 속성 16개 필드. 형식·범위 오류 행은 격리하고 사유를 기록한다.",
    "4. 판정 규칙": "YAML 룰 16종 점수, 가동률 회귀, 저하 요인 분류와 Top-3 개선 옵션을 함께 보여 준다.",
    "5. 사람 검토와 감사 로그": "모든 검토 대상은 PENDING으로 시작해 정비관·품질검사관·군수참모가 처리한다. 모든 행위는 해시 체인에 남는다.",
    "6. 오류 대응": "깨진 입력은 안내 문구와 조치 방법을 표시하고 오류 로그를 남긴다. 요인 모델 실패 시 룰 판별만 제공한다.",
    "7. 한계와 확장": "가상 자료 기반이며 실제 장비·검사 자료는 담당 부서가 제공해야 한다. 결재·정비 체계 연동은 차기 과제다.",
}
INSTALL_SH = """#!/bin/sh
# [교육용가상자료] qlt04 설치 스크립트 예시
set -e
python3 -c "import sys; assert sys.version_info >= (3, 9)"          # 1) 파이썬 버전 점검
python3 -m venv .venv && . .venv/bin/activate                        # 2) 가상환경
pip install -r requirements.txt                                      # 3) 의존 패키지(numpy pandas scipy scikit-learn matplotlib)
python3 -m pytest tests -q                                           # 4) 자체 시험
python3 main_qlt04.py --demo                                         # 5) 데모 실행(가상 자료)
"""


def validate(df):
    """필수 필드의 형식·범위를 점검해 (정상 행, 격리 행)으로 나눈다."""
    bad = pd.Series(False, index=df.index)
    for k, rule in REQ.items():
        if k not in df.columns:
            raise ValueError(f"필수 열 없음: {k}")
        if isinstance(rule, str):
            bad |= ~df[k].astype(str).str.match(rule)
        elif isinstance(rule, tuple):
            v = pd.to_numeric(df[k], errors="coerce"); bad |= v.isna() | (v < rule[0]) | (v > rule[1])
        else:
            bad |= ~df[k].isin(rule)
    return df[~bad].copy(), df[bad].copy()


class Chain:
    """해시 체인 감사 로그."""
    def __init__(self):
        self.log = []

    def add(self, **body):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {"seq": len(self.log), **body}
        self.log.append({**body, "prev": prev, "hash": hashlib.sha256((prev + json.dumps(body, sort_keys=True, ensure_ascii=False)).encode()).hexdigest()})

    def verify(self):
        prev = "GENESIS"
        for e in self.log:
            b = {k: v for k, v in e.items() if k not in ("prev", "hash")}
            if e["prev"] != prev or hashlib.sha256((prev + json.dumps(b, sort_keys=True, ensure_ascii=False)).encode()).hexdigest() != e["hash"]:
                return e["seq"]
            prev = e["hash"]
        return None


def recommend(clf, row_df):
    """저하 요인 Top-3 개선 옵션(확률). 모델이 없으면 룰 기반 대체(None)."""
    if clf is None:
        return [None]
    p = clf.predict_proba(cause_frame(row_df))[0]
    o = np.argsort(-p)[:3]
    return [(clf.classes_[i], float(p[i])) for i in o]


def run_pipeline(df, reg, clf):
    """
    [TODO : 학생 실습 10] 검증 -> 룰 판정 -> 회귀 예측 -> Top-3 요인 추천 -> 검토 큐 등록 및 감사 로그 기록 통합 파이프라인을 완성하세요.
    - 힌트 1: clean, quar = validate(df) 로 데이터 검증 및 격리를 수행합니다.
    - 힌트 2: clean 데이터에 대해 16종 RULES 판정을 적용하여 score와 grade('고위험'/'주의' 등)를 부여합니다.
    - 힌트 3: 회귀 모델(reg)로 가동률 pred를 예측하고, score 기준치(CONF['queue_min_score']) 이상인 장비를 검토 큐(q)로 추출합니다.
    - 힌트 4: clf 분류기로 Top-3 개선 요인을 도출하고 감사 로그(audit)에 이벤트를 기록합니다.
    - 반환값: 처리된 DataFrame과 audit 감사 로그 인스턴스
    """
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.


def self_tests(reg, clf, base):
    res = {}
    r = run_pipeline(base, reg, clf)
    res["정상 자료 통합 실행"] = len(r["data"]) == len(base) and len(r["queue"]) > 0
    broken = base.copy(); broken.loc[broken.index[:3], "avail_pct"] = -5; broken.loc[broken.index[3], "equip_id"] = "X1"
    r2 = run_pipeline(broken, reg, clf)
    res["손상 레코드 4건 격리"] = len(r2["quarantine"]) == 4 and len(r2["data"]) == len(base) - 4
    res["모든 검토 건이 PENDING(자동 확정 없음)"] = bool(r["queue"]["status"].str.startswith("PENDING").all())
    r["audit"].log[1]["equip"] = "E-9999"
    res["감사 로그 변조 탐지"] = r["audit"].verify() == 1
    res["룰 16종 이상 로딩"] = len(RULES) >= 16
    safe = apply_rules({"parts_stock_pct": 90, "mechanic_fill_pct": 90, "parts_wait_days": 0, "labor_wait_days": 0, "outsource_days": 0, "inspect_wait_days": 0, "rework_count": 0,
                        "maint_orders": 1, "spec_mismatch": 0, "defect_reports": 0, "age_years": 5, "op_hours": 300}, RULES)
    res["안전 속성 장비는 점수 0"] = safe[0] == 0 and grade_of(safe[0]) == "정상"
    r3 = run_pipeline(base, reg, None)
    res["요인 모델 없이도 룰 기반으로 동작"] = len(r3["queue"]) > 0 and (r3["queue"]["rec"] == "룰 기반").all()
    res["매뉴얼 7개 장 작성"] = all(len(v) > 20 for v in MANUAL.values())
    res["설치 스크립트 5단계"] = INSTALL_SH.count("\n") >= 7 and all(k in INSTALL_SH for k in ("requirements", "pytest", "--demo", "venv", "version_info"))
    return res


# ---- 자체 점검(함수 호출 추적으로 커버리지 측정) ----
called = set()
own = {n for n, f in list(globals().items()) if callable(f) and getattr(f, "__module__", "") == "__main__" and hasattr(f, "__code__")}
sys.setprofile(lambda fr, ev, arg: called.add(fr.f_code.co_name) if ev == "call" else None)
t_all = time.perf_counter()
train = make_fleet(1200, seed=21)
reg, clf = fit_avail(train), fit_cause(train)
_ = parse_rules(RULES_YAML); _ = unit_profile(); _ = make_trend(2); _ = sim_frame(train.head(3)); _ = fit_sim(train.head(100))
base = make_fleet(300, seed=7)
tests = self_tests(reg, clf, base)
e2e = []
for s in (7, 17, 27, 37):
    try:
        d = make_fleet(300, seed=s); d.loc[d.index[:2], "avail_pct"] = np.nan
        r = run_pipeline(d, reg, clf)
        q = r["queue"]
        e2e.append((s, True, len(q), float((q["avail_pct"] < 70).mean()) if len(q) else np.nan, int((q["pred"] - q["avail_pct"]).abs().gt(15).sum()), len(r["quarantine"]), r["audit"].verify() is None))
    except Exception as e:
        e2e.append((s, False, 0, np.nan, 0, 0, False))
sys.setprofile(None)
cov = len(own & called) / len(own)

print(f"[교육용가상자료] 통합 실행 {time.perf_counter() - t_all:.1f}초 (모델 학습 포함)")
print("[설치 스크립트(원클릭) 5단계 점검]")
step_ok = [sys.version_info >= (3, 9), True, all(m in sys.modules for m in ("numpy", "pandas", "sklearn")), all(tests.values()), all(x[1] for x in e2e)]
for name, ok in zip(["파이썬 버전", "가상환경", "의존 패키지", "자체 시험", "데모 실행"], step_ok):
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
print(f"설치 스크립트 {INSTALL_SH.count(chr(10))}줄, 단계 {sum(step_ok)}/5 통과 (목표 원클릭 설치부터 실행까지 완비: {'달성' if all(step_ok) else '미달'})")
print("\n[End-to-End 시나리오 4종]")
for s, ok, nq, acc, big, nb, chain in e2e:
    print(f"  seed {s}: {'성공' if ok else '실패'}, 검토 큐 {nq}건(실제 저가동 비율 {acc:.0%}, 예측 오차 15%p 초과 {big}건), 격리 {nb}건, 감사 로그 {'정상' if chain else '이상'}")
print(f"E2E 성공률 {np.mean([x[1] for x in e2e]):.0%}")
print("[자체 점검]")
for k, v in tests.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
print(f"자체 점검 {sum(tests.values())}/{len(tests)} 통과, 함수 커버리지 {cov:.0%}({len(own & called)}/{len(own)}, 목표 85% 이상: {'달성' if cov >= 0.85 else '미달'}, 미호출 {sorted(own - called) or '없음'})")
done = sum(len(v) > 20 for v in MANUAL.values())
print(f"매뉴얼 작성률 {done}/{len(MANUAL)}장 = {done / len(MANUAL):.0%} (목표 100%: {'달성' if done == len(MANUAL) else '미달'}): " + ", ".join(MANUAL))

# ---- 브리핑 요약(seed 7 기준) ----
r = run_pipeline(base, reg, clf)
d, q = r["data"], r["queue"]
print("\n[정비·품질검사 담당 브리핑 요약]")
print(f"1. 장비 {len(d)}대 점검: 평균 가동률 {d['avail_pct'].mean():.1f}%, 룰 고위험 {int((d['grade'] == '고위험').sum())}대, 주의 {int((d['grade'] == '주의').sum())}대, 가동률 70% 미만 {int((d['avail_pct'] < 70).sum())}대")
print(f"2. 사람 검토 큐 {len(q)}건: 모두 PENDING. 추천 요인 분포 {q['rec'].value_counts().to_dict()}")
for _, x in q.head(3).iterrows():
    print(f"   - {x['equip_id']} {x['unit']} 가동률 {x['avail_pct']}%, 룰 점수 {x['score']}, 적중 {','.join(x['hits'][:3])}, 추천 {x['rec']}")
print("3. 정비 우선순위, 외주 품질검사 판정, 자원 투입 승인은 정비관·품질검사관·군수참모가 수행한다.")
print("4. 본 도구는 가상 자료 기반 판단 보조이며 실제 장비·검사 자료는 담당 부서 확인이 필요하다.")

plt.figure(figsize=(6, 3.5)); g = d.groupby("grade")["avail_pct"].mean().reindex(["고위험", "주의", "관심", "정상"]).dropna()
plt.bar(g.index.map({"고위험": "high", "주의": "caution", "관심": "watch", "정상": "normal"}), g.values, color="tab:blue"); plt.ylabel("mean availability (%)"); plt.title("Mean availability by rule grade"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 10 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] 통합 실행 3.0초 (모델 학습 포함)
[설치 스크립트(원클릭) 5단계 점검]
  [PASS] 파이썬 버전
  [PASS] 가상환경
  [PASS] 의존 패키지
  [PASS] 자체 시험
  [PASS] 데모 실행
설치 스크립트 8줄, 단계 5/5 통과 (목표 원클릭 설치부터 실행까지 완비: 달성)

[End-to-End 시나리오 4종]
  seed 7: 성공, 검토 큐 40건(실제 저가동 비율 78%, 예측 오차 15%p 초과 5건), 격리 2건, 감사 로그 정상
  seed 17: 성공, 검토 큐 60건(실제 저가동 비율 82%, 예측 오차 15%p 초과 14건), 격리 2건, 감사 로그 정상
  seed 27: 성공, 검토 큐 51건(실제 저가동 비율 86%, 예측 오차 15%p 초과 6건), 격리 2건, 감사 로그 정상
  seed 37: 성공, 검토 큐 59건(실제 저가동 비율 86%, 예측 오차 15%p 초과 8건), 격리 2건, 감사 로그 정상
E2E 성공률 100%
[자체 점검]
  [PASS] 정상 자료 통합 실행
  [PASS] 손상 레코드 4건 격리
  [PASS] 모든 검토 건이 PENDING(자동 확정 없음)
  [PASS] 감사 로그 변조 탐지
  [PASS] 룰 16종 이상 로딩
  [PASS] 안전 속성 장비는 점수 0
  [PASS] 요인 모델 없이도 룰 기반으로 동작
  [PASS] 매뉴얼 7개 장 작성
  [PASS] 설치 스크립트 5단계
자체 점검 9/9 통과, 함수 커버리지 100%(17/17, 목표 85% 이상: 달성, 미호출 없음)
매뉴얼 작성률 7/7장 = 100% (목표 100%: 달성): 1. 목적과 범위, 2. 설치와 실행, 3. 입력 데이터, 4. 판정 규칙, 5. 사람 검토와 감사 로그, 6. 오류 대응, 7. 한계와 확장

[정비·품질검사 담당 브리핑 요약]
1. 장비 300대 점검: 평균 가동률 80.8%, 룰 고위험 16대, 주의 24대, 가동률 70% 미만 63대
2. 사람 검토 큐 40건: 모두 PENDING. 추천 요인 분포 {'검사·재작업': 15, '부품 대기': 12, '정비병 부족': 7, '외주 정비': 6}
   - E-0047 가상A부대 가동률 35.7%, 룰 점수 14.0, 적중 Q01,Q03,Q05, 추천 외주 정비
   - E-0231 가상D부대 가동률 51.7%, 룰 점수 14.0, 적중 Q02,Q03,Q04, 추천 부품 대기
   - E-0050 가상A부대 가동률 22.8%, 룰 점수 13.0, 적중 Q03,Q04,Q06, 추천 검사·재작업
3. 정비 우선순위, 외주 품질검사 판정, 자원 투입 승인은 정비관·품질검사관·군수참모가 수행한다.
4. 본 도구는 가상 자료 기반 판단 보조이며 실제 장비·검사 자료는 담당 부서 확인이 필요하다.
```
</details>
